In [11]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bioio import BioImage
import os
import numpy as np
from dask.distributed import Client, get_client
import napari
import xmltodict

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
from skimage import io
import glob
from pathlib import Path

import ezomero
from omero.model.enums import UnitsLength
import omero

In [2]:
viewer = napari.Viewer()

In [3]:
try:
  client = get_client()
except ValueError:
  client = Client()

print(client.dashboard_link)

http://127.0.0.1:8787/status


In [4]:
root = r'E:\UserData\Johannes\20260420\2026_04_21_09_49_52--TRG1970_001\TileScan 1'
file_names = glob.glob(os.path.join(root, "**", "*ome.tif"), recursive=True)
file_names

['E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\10.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\3.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\4.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\5.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\6.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\7.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\8.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\9.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\D\\10.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\D\\3.ome.tif',
 'E:\\UserData\\Johannes\\20

In [5]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

In [6]:
write_jobs = []
images = []
transforms = []
for filename in file_names:
	img_basename = Path(filename).stem.split(".")[0] 
	outfile_oz = img_basename + ".ome.zarr"
	bf = BioImage(filename)
	

	if bf.ome_metadata.images[0].pixels.physical_size_z is None:
		size_z = 0
	else:
		size_z = bf.ome_metadata.images[0].pixels.physical_size_z

	scale = {
		"z": size_z,
		"y": bf.ome_metadata.images[0].pixels.physical_size_y,
		"x": bf.ome_metadata.images[0].pixels.physical_size_x
	}

	axes_units = {
		"z": "micrometer",
		"y": "micrometer",
		"x": "micrometer",
	}
	
	# parse tile metadata
	metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{img_basename}.xlif")
	metadata_dict = xmltodict.parse(open(metadata_file).read())
	tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

	# n_series= bf.series_count()
	n_series = len(bf.scenes)

	all_positions = []
	if n_series > 1:
		sims = []
		for idx in range(n_series):
			bf.set_scene(bf.scenes[idx])
			dask_image = bf.dask_data
			size_tile = np.asarray(dask_image.squeeze().shape) * np.asarray([scale["z"], scale["y"], scale["x"]])
			# dask_image = bf.to_dask(series=idx)
			ts_info = tile_scan_info[idx]

			t_z = float(ts_info["@PosZ"])
			t_y = float(ts_info["@PosY"])
			t_x = float(ts_info["@PosX"])

			all_positions.append(np.asarray([t_z, t_y, t_x]))

			sims.append(
				si_utils.get_sim_from_array(
					dask_image,
					dims=["t", "c", "z", "y", "x"],
					scale=scale,
					translation={"z": t_z, "y": t_y, "x": t_x},
					transform_key="stage_metadata",
					)
			)

		image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)
	else:
		# image = bf.to_dask(series=0)
		image = bf.dask_data
		size_tile = np.asarray(dask_image.squeeze()) * np.asarray([scale["z"], scale["y"], scale["x"]])
		t_z = float(tile_scan_info["@PosZ"])
		t_y = float(tile_scan_info["@PosY"])
		t_x = float(tile_scan_info["@PosX"])

		all_positions.append(np.asarray([t_z, t_y, t_x]))

	all_positions = np.stack(all_positions, axis=0)
	position_corner = np.min(all_positions, axis=0) - size_tile / 2

	ngff_image = NgffImage(
		image.squeeze(),
		dims=["z", "y", "x"],
		scale=scale,
		axes_units=axes_units,
		name=Path(filename).parent.stem + img_basename
	)
	ngff_multiscales = NgffMultiscales(
		image=ngff_image,
	)
	images.append(ngff_multiscales)

	transform = Translation(
		translation=position_corner.tolist(),
		input=CoordinateSystemIdentifier(
			path=ngff_multiscales.name,
			name=ngff_multiscales.metadata.coordinateSystems[0].name
		),
		output=CoordinateSystemIdentifier(
			name="world"
		)
	)
	transforms.append(transform)

In [7]:
cs_world = CoordinateSystem(
	name="world",
	axes=(
		Axis(name="z", type="space", unit="micrometer"),
		Axis(name="y", type="space", unit="micrometer"),
		Axis(name="x", type="space", unit="micrometer"),
	)
)

ngff_scene = NgffScene(
	images=images,
	transformations=transforms,
	coordinate_systems=(
		cs_world,
	)
)

delayed_objects = ngff_scene.to_ome_zarr(
	os.path.join(root, "scene.ome.zarr")
)

## Show scenes



In [8]:
viewer.open(os.path.join(root, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'C10'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314255, 0.08876316990000001]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'C3'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314255, 0.0257631711]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'C4'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314255, 0.0347631716]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'C5'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314255, 0.0437631692]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'C6'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314255, 0.052763171]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'C7'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.0237314

[<Image layer 'scene.ome' at 0x1d4ddf60980>,
 <Image layer 'scene.ome [1]' at 0x1d4ddf80140>,
 <Image layer 'scene.ome [2]' at 0x1d4ddf809e0>,
 <Image layer 'scene.ome [3]' at 0x1d4ddf82c90>,
 <Image layer 'scene.ome [4]' at 0x1d4e9a14170>,
 <Image layer 'scene.ome [5]' at 0x1d4eba3e450>,
 <Image layer 'scene.ome [6]' at 0x1d4ec6ba2a0>,
 <Image layer 'scene.ome [7]' at 0x1d4e7b349e0>,
 <Image layer 'scene.ome [8]' at 0x1d4e4f54410>,
 <Image layer 'scene.ome [9]' at 0x1d4e4f54830>,
 <Image layer 'scene.ome [10]' at 0x1d4e85f01a0>,
 <Image layer 'scene.ome [11]' at 0x1d4d8cbda90>,
 <Image layer 'scene.ome [12]' at 0x1d4e92d0830>,
 <Image layer 'scene.ome [13]' at 0x1d4eb21dd00>,
 <Image layer 'scene.ome [14]' at 0x1d4f05ddf70>,
 <Image layer 'scene.ome [15]' at 0x1d4e656db80>,
 <Image layer 'scene.ome [16]' at 0x1d4f07d7e30>,
 <Image layer 'scene.ome [17]' at 0x1d4e64a48c0>,
 <Image layer 'scene.ome [18]' at 0x1d4e13e2120>,
 <Image layer 'scene.ome [19]' at 0x1d4eaf27110>,
 <Image layer 

## Optional: Upload tiles to OMERO

In [12]:
host = 'omero-nfdi.uni-muenster.de'
port = 4064
user = 'JohannesSoltwedel'
group = None
secure=True
password = "BYMCfyZpMDLC"

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

In [ ]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(2560, 2560)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (2560, 2560):
			continue

		tile = chunk.compute()

		if sum(tile.flatten()==1) > 10000:
			continue

		image_id = ezomero.post_image(
			conn,
			image=tile[None, None, None],
			dataset_id=251,
			image_name=f"{layer.name}_chunk_{idx}", dim_order='tczyx', across_groups=False)
		
		conn.setChannelNames(data_type='Image', ids=[image_id], nameDict={1: 'Brightfield'})

		unit_x = omero.model.LengthI(scale["x"]* 10e6, UnitsLength.MICROMETER)
		unit_y = omero.model.LengthI(scale["y"]* 10e6, UnitsLength.MICROMETER)

		image_object = conn.getObject("Image", image_id)
		primary_pixels_object = image_object.getPrimaryPixels()._obj
		primary_pixels_object.setPhysicalSizeX(unit_x)
		primary_pixels_object.setPhysicalSizeY(unit_y)
		conn.getUpdateService().saveObject(primary_pixels_object)


KeyboardInterrupt: 

In [29]:
1280 * 2

2560